# 02. Prompt Engineering 실습
**SK하이닉스 Autonomous R&D — Day 3** 

---
## 0. 준비

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv('OPEN_API_KEY'))


def ask(system_prompt, user_prompt, temperature=0.2):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

In [2]:
# 기본적으로 같은 모델이라도 프롬프트에 따라 다른 출력을 생성

system = 'You are a helpful assistant. Answer in Korean.'

prompts_ko = [
    "프랑스의 수도는",
    "Q: 프랑스의 수도는 어디인가요? A:",
    "프랑스의 수도 도시는",
]

for prompt in prompts_ko:
    print(f"프롬프트: {prompt}")
    print(f"생성: {ask(system, prompt, temperature=0.2)}")
    print()

프롬프트: 프랑스의 수도는
생성: 프랑스의 수도는 파리입니다.

프롬프트: Q: 프랑스의 수도는 어디인가요? A:
생성: 프랑스의 수도는 파리입니다.

프롬프트: 프랑스의 수도 도시는
생성: 프랑스의 수도 도시는 파리입니다.



---
## 1. Zero-shot vs Few-shot 

| 방식 | 설명 |
|------|------|
| **Zero-shot** | 예시 없이 지시만 |
| **Few-shot** | 원하는 형식의 **예시**를 함께 제공 |

In [ ]:
# 

In [3]:
system = 'You are a helpful assistant. Answer in Korean.'
# Zero-shot — 형식 지시 없음
user_zero = '''
아래 영화 리뷰가 긍정인지 부정인지 판정해줘.
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Zero-shot ===')
print(ask(system, user_zero))

=== Zero-shot ===
첫 번째 리뷰는 부정적인 요소가 포함되어 있어 부정으로 판정할 수 있습니다. 두 번째 리뷰는 긍정적인 감정을 표현하고 있어 긍정으로 판정할 수 있습니다.


In [4]:
# Few-shot — 출력 형식 예시 포함
user_few = '''
아래 형식으로 감성 판정해줘.

[예시]
입력: "배우 연기가 훌륭했다" → 긍정 | 9/10
입력: "스토리가 지루했다" → 부정 | 3/10

[실제 데이터]
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Few-shot ===')
print(ask(system, user_few))

=== Few-shot ===
입력: "연출은 좋았지만 2시간이 너무 길었다" → 긍정 | 6/10  
입력: "최고의 영화, 또 보고 싶다" → 긍정 | 10/10


In [5]:
zero_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

문제: 행복한 : 슬픈 :: 좋은 :"""

print(ask(system, zero_shot_prompt_ko, temperature=0.7))

좋은 : 나쁜


In [6]:
few_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :"""

print('Few-shot 프롬프트:')
print(few_shot_prompt_ko)
print('\n생성된 완성:')
print(ask(system, few_shot_prompt_ko, temperature=0.7))

Few-shot 프롬프트:
다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :

생성된 완성:
나쁜


---
## 2. 역할(Role) 설정 

In [7]:
question = '하루 커피 4잔 마셔도 괜찮을까?'

print('=== 일반 assistant ===')
print(ask('You are a helpful assistant.', question))
print('\n' + '=' * 50 + '\n')

=== 일반 assistant ===
하루에 커피 4잔을 마시는 것은 대부분의 사람들에게 일반적으로 괜찮습니다. 연구에 따르면, 성인은 하루에 400mg 이하의 카페인을 섭취하는 것이 안전하다고 여겨지며, 이는 대략 커피 4잔에 해당합니다. 하지만 개인의 카페인 민감도, 건강 상태, 임신 여부 등에 따라 다를 수 있습니다.

카페인 섭취가 불안, 불면증, 심장 두근거림 등을 유발할 수 있으므로, 이러한 증상이 나타난다면 섭취량을 줄이는 것이 좋습니다. 또한, 특정 건강 문제가 있는 경우에는 의사와 상담하는 것이 바람직합니다.




In [ ]:
print('=== 영양사 역할 ===')
print(ask(
    'You are a registered dietitian. Answer in Korean with caffeine mg estimate and health advice.',
    question,
))

In [8]:
roles = [
    "당신은 친절한 고객 서비스 상담원입니다.",
    "당신은 전문적인 기술 문서 작성자입니다.",
    "당신은 창의적인 소설가입니다."
]

question = "인공지능에 대해 설명해주세요."

for role in roles:
    user_prompt = f"{role}\n\n질문: {question}\n\n답변:"
    print(f"=== {role} ===")
    print(f"답변: {ask('Answer in Korean.', user_prompt, temperature=0.8)}")
    print()

=== 당신은 친절한 고객 서비스 상담원입니다. ===
답변: 안녕하세요! 인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 문제 해결, 의사 결정 등을 수행할 수 있도록 하는 기술입니다. 인공지능은 데이터 분석과 알고리즘을 기반으로 하여, 경험을 통해 스스로 발전하고 개선할 수 있는 능력을 가지고 있습니다.

AI는 크게 두 가지로 나눌 수 있습니다. 첫째는 좁은 인공지능(Narrow AI)으로, 특정 작업이나 문제를 해결하는 데 특화된 시스템입니다. 예를 들어, 음성 인식, 이미지 분석, 자율주행차 등이 이에 해당합니다. 둘째는 일반 인공지능(General AI)으로, 인간과 유사한 수준의 지능을 가지고 여러 가지 작업을 수행할 수 있는 AI를 말합니다. 현재 일반 인공지능은 아직 개발되지 않았습니다.

인공지능은 다양한 분야에서 활용되고 있으며, 의료, 금융, 교육, 제조업 등에서 혁신을 이루고 있습니다. 앞으로도 AI 기술은 계속 발전하여 우리의 생활을 더욱 편리하게 만들어 줄 것으로 기대됩니다. 추가적으로 궁금한 점이 있으시면 언제든지 말씀해 주세요!

=== 당신은 전문적인 기술 문서 작성자입니다. ===
답변: 인공지능(Artificial Intelligence, AI)은 컴퓨터 시스템이 인간의 지능을 모방하거나 인간처럼 사고하고 학습할 수 있도록 하는 기술과 이론을 포괄하는 분야입니다. AI는 문제 해결, 패턴 인식, 자연어 처리, 이미지 인식 등 다양한 작업을 수행할 수 있도록 설계되었습니다.

AI는 크게 두 가지로 나눌 수 있습니다:

1. **약한 인공지능(Weak AI)**: 특정 작업을 수행하도록 설계된 시스템으로, 인간의 지능을 완전히 대체하지는 않지만 특정 문제에 대해 뛰어난 성능을 보입니다. 예를 들어, 음성 인식 소프트웨어, 추천 시스템 등이 있습니다.

2. **강한 인공지능(Strong AI)**: 인간의 사고 능력을 완전히 모방할 수 있는 인공지능으로, 현재 연구 단계에 있으며, 아직 구현된 사례는 없습니다.

---
## 3. 출력 형식 지정 

In [9]:
topic = '재택근무의 장단점 3가지'
system_ko = 'Answer in Korean.'

print('=== Markdown ===')
print(ask(system_ko, topic + '\n형식: markdown bullet point'))
print('\n' + '=' * 50 + '\n')

=== Markdown ===
### 재택근무의 장단점

#### 장점
- **유연한 근무 시간**: 개인의 일정에 맞춰 근무 시간을 조정할 수 있어 일과 생활의 균형을 맞추기 용이함.
- **교통비 및 시간 절약**: 출퇴근이 필요 없어 교통비와 시간을 절약할 수 있으며, 스트레스도 감소함.
- **편안한 근무 환경**: 자신이 선호하는 환경에서 일할 수 있어 집중력과 생산성이 향상될 수 있음.

#### 단점
- **사회적 고립감**: 동료와의 직접적인 소통이 줄어들어 외로움을 느낄 수 있으며, 팀워크가 약화될 수 있음.
- **일과 개인 생활의 경계 모호**: 집에서 일하다 보면 업무와 개인 생활의 경계가 흐려져 과중한 업무에 시달릴 수 있음.
- **자기 관리 필요**: 스스로 동기부여를 해야 하며, 자율성이 요구되기 때문에 자기 관리가 어려운 경우 생산성이 떨어질 수 있음.




In [10]:
print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (항목 | 장점 | 단점)만 출력',
))

=== Table ===
| 항목       | 장점                       | 단점                       |
|------------|----------------------------|----------------------------|
| 유연성     | 근무 시간 조정 가능        | 일과 생활의 경계 모호       |
| 교통비 절감 | 출퇴근 시간 및 비용 절감   | 사회적 고립감 증가          |
| 생산성     | 집중할 수 있는 환경 제공   | 자율성 부족 시 생산성 저하  |


---
### 출력 형식 (표 / JSON)

In [ ]:
import json

topic = '식각(Etch) 공정에서 수율에 영향 주는 요인 3가지'
system_ko = 'Answer in Korean.'

print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (요인 | 설명 | 대응)만 출력',
))

In [11]:
print('=== JSON ===')
result = ask(
    'Output ONLY valid JSON. No markdown.',
    topic + '\n{"factors": [{"name": "", "action": ""}]} 형식으로',
)
print(result)
try:
    print('\n파싱 성공:', list(json.loads(result).keys()))
except json.JSONDecodeError:
    print('\n파싱 실패 — ONLY valid JSON 강조 필요')

=== JSON ===
{
  "factors": [
    {
      "name": "유연한 근무 시간",
      "action": "재택근무는 근무 시간을 조정할 수 있어 개인의 생활 패턴에 맞출 수 있다."
    },
    {
      "name": "교통비 절감",
      "action": "출퇴근이 필요 없어 교통비와 시간을 절약할 수 있다."
    },
    {
      "name": "집중력 향상",
      "action": "자신의 작업 환경을 조절할 수 있어 집중력을 높일 수 있다."
    },
    {
      "name": "사회적 고립",
      "action": "동료와의 대면 소통이 줄어들어 사회적 고립감을 느낄 수 있다."
    },
    {
      "name": "업무와 개인 생활의 경계 모호",
      "action": "업무와 개인 생활의 경계가 흐려져 스트레스를 받을 수 있다."
    },
    {
      "name": "자기 관리 필요",
      "action": "자기 주도적으로 업무를 관리해야 하므로 자기 관리 능력이 요구된다."
    }
  ]
}


NameError: name 'json' is not defined

---
## 4. Chain-of-Thought (CoT)

CoT는 모델이 단계별로 추론 과정을 보여주도록 하는 기법.

In [12]:
## zero-shot 예시

problem = '''
A팀 10명, B팀 15명, C팀 5명.
전체 25명 중 40% 이상이 A팀이면 A팀에 추가 인원 배치.
지금 추가 배치가 필요한가?
'''
system = 'You are a helpful assistant. Answer in Korean.'
print('=== CoT 없음 ===')
print(ask(system, problem))

=== CoT 없음 ===
전체 25명 중 40%는 10명입니다. 현재 A팀은 10명으로, 전체 인원의 40%에 해당합니다. 따라서 A팀에 추가 배치가 필요하지 않습니다. A팀의 인원이 40% 이상이 되려면 11명 이상이어야 합니다.


In [13]:
print('=== CoT 적용 ===')
print(ask(
    system + ' Show step-by-step reasoning before final answer.',
    problem + '\n\n단계별로 생각해 봅시다.',
))

=== CoT 적용 ===
전체 인원은 25명입니다. A팀, B팀, C팀의 인원 수는 다음과 같습니다:

- A팀: 10명
- B팀: 15명
- C팀: 5명

1. **A팀의 비율 계산**:
   A팀의 인원 수는 10명입니다. 전체 인원 25명 중 A팀의 비율을 계산해 보겠습니다.
   \[
   \text{A팀 비율} = \frac{\text{A팀 인원}}{\text{전체 인원}} \times 100 = \frac{10}{25} \times 100 = 40\%
   \]

2. **40% 이상인지 확인**:
   A팀의 비율이 40%입니다. 문제에서 요구하는 것은 A팀의 인원이 40% 이상일 경우 추가 인원 배치가 필요하다고 했습니다. 현재 A팀의 비율은 정확히 40%입니다.

3. **결론**:
   A팀의 비율이 40% 이상이므로, 추가 배치가 필요합니다.

따라서, A팀에 추가 인원 배치가 필요합니다.


In [14]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

답: 3,600원


In [15]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

천천히 단계별로 생각해봅시다.

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

좋습니다. 문제를 단계별로 해결해봅시다.

1. 연필의 가격 계산:
   - 연필 1자루의 가격은 500원입니다.
   - 연필 5자루의 가격은 500원 × 5 = 2500원입니다.

2. 지우개의 가격 계산:
   - 지우개 1개의 가격은 300원입니다.
   - 지우개 3개의 가격은 300원 × 3 = 900원입니다.

3. 총 금액 계산:
   - 연필과 지우개의 총 금액은 2500원 + 900원 = 3400원입니다.

따라서, 총 금액은 3400원입니다.

답: 3400원


In [16]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

답: 6000원

---

문제: 한 상점에서 귤 1개와 바나나 12개를 샀습니다. 사과는 개당 1000원, 배는 개당 200원입니다. 총 금액은 얼마인가요?

답: 3400원

---


문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: """

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

답: 3900원


In [17]:
## one-shot 예시

problem = """다음 문제를 단계별로 풀어보세요.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

단계별 풀이:
1. 사과 3개의 가격: 3 × 1000 = 3000원
2. 배 2개의 가격: 2 × 1500 = 3000원
3. 총 금액: 3000 + 3000 = 6000원
답: 6000원

---

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

단계별 풀이:"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

단계별 풀이:

1. 연필 5자루의 가격: 5 × 500 = 2500원
2. 지우개 3개의 가격: 3 × 300 = 900원
3. 총 금액: 2500 + 900 = 3400원

답: 3400원


---
## 5. System + User 프롬프트 

In [18]:
system_prompt = '''
너는 스타트업 PM의 일정 관리 AI다.
규칙: 결론 먼저, bullet 3개, 한국어
'''

user_prompt = '''
이번 주 마감: 월-기획안, 수-코드리뷰, 금-발표.
목요일에 하루 종일 회의가 잡혔어. 우선순위 조정해줘.
'''

print(ask(system_prompt, user_prompt))

결론: 목요일 회의에 맞춰 우선순위를 조정하여 월요일에 기획안 제출, 수요일에 코드 리뷰, 금요일에 발표 준비를 완료하세요.

- 월요일: 기획안 제출
- 수요일: 코드 리뷰
- 금요일: 발표 준비 및 최종 점검


In [19]:
system_md = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤 "
    "리스크와 가정을 명시해야 한다."
)

user_md = "이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘."

answer = ask(system_md, user_md, temperature=0.7)
print(answer)

| 결론 | VIP 이탈 방지를 위한 액션 플랜 |
|------|-------------------------------|
| VIP 고객의 이탈을 최소화하기 위해 맞춤형 리워드 프로그램과 개인화된 커뮤니케이션을 강화해야 한다. |

| 리스크 | 가정 |
|--------|------|
| 1. 고객의 반응이 미미할 수 있다. | 1. VIP 고객이 개인화된 접근을 선호한다. |
| 2. 비용 증가의 우려가 있다. | 2. 리워드 프로그램이 이탈 방지에 효과적일 것이다. |
| 3. 데이터 분석의 정확성 문제. | 3. VIP 고객의 이탈 원인을 정확히 파악할 수 있을 것이다. |

### 액션 플랜 세부사항

| 단계 | 액션 아이템 | 책임자 | 기한 | 비고 |
|------|--------------|--------|------|------|
| 1    | VIP 고객 데이터 분석 | 데이터 팀 | 이번 주 | 이탈 원인 파악 |
| 2    | 맞춤형 리워드 프로그램 설계 | 마케팅 팀 | 다음 주 | 고객 선호도 반영 |
| 3    | 개인화된 커뮤니케이션 시작 | 고객 서비스 팀 | 다음 주 | 이메일 및 문자 발송 |
| 4    | 피드백 수집 및 분석 | 고객 서비스 팀 | 2주 후 | 프로그램 효과 검토 |
| 5    | 프로그램 조정 및 개선 | 전체 팀 | 1개월 후 | 지속적인 개선 필요 |


---
## 6. Self-check Prompting

In [ ]:
check_prompt = f"""아래는 네가 작성한 'VIP 이탈 상위 200명 액션 플랜' 초안이다.

[초안]
{answer}

이제 아래 체크리스트로 점검하고, 필요하면 수정본을 작성하라.

체크리스트:
1) 논리적 모순/비약이 있는가?
2) 실행 불가능하거나 모호한 액션이 있는가? (담당/기한/방법이 불명확한지)
3) 과도한 가정이 있는가?
4) 표 형식이 지켜졌는가? (결론 먼저, 리스크/가정 포함)

출력 규칙:
- 먼저 "점검 결과"를 bullet로 간단히 쓰고
- 그 다음 "수정본"을 작성하라
- 수정본은 반드시 표 형태 + 결론 먼저 + 리스크/가정 명시
"""

final_answer = ask(system_md, check_prompt, temperature=0.7)
print('\n--- Self-check 후 최종 답변(수정본) ---')
print(final_answer)

---
## 7. Constraint Prompting

In [ ]:
system_prompt = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤, "
    "리스크와 가정을 명시해야 한다."
)

constraints = """조건(반드시 준수):
- 추가 예산은 최대 1억 원 이내
- 인력 증원 불가 (기존 인력 내에서 운영)
- 2주 이내 실행 가능한 액션만 제시
- 고객에게 직접 연락하는 액션은 과도한 빈도를 피하고(1회 이내),
  개인정보/컴플라이언스 리스크를 고려할 것
"""

user_prompt = f"""이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘.

{constraints}

출력 형식:
- 결론(한 줄) 먼저
- 그 다음 표로 정리 (컬럼 예: 타깃/액션/채널/우선순위/기간/기대효과/담당)
- 마지막에 리스크/가정 명시
"""

constrained_answer = ask(system_prompt, user_prompt, temperature=0.7)
print('\n--- 조건 기반 최종 답변(답변만) ---')
print(constrained_answer)